# 🏍️ PotholeGuard — AI Pothole Detection Model (YOLO11n)
### Complete Google Colab GPU Training, Evaluation, & ONNX Export Pipeline

This notebook trains and validates the single-class **YOLO11n** pothole detection model for **PotholeGuard**.

---
### Pipeline Stages:
1. **Environment Setup & GPU Verification** (T4 GPU)
2. **Dataset Acquisition & Organization** (RDD2022 India / Roboflow Universe)
3. **Data Validation & Sanity Inspection** (`dataset_check.py`)
4. **YOLO11n Model Training** (Road-optimized augmentations, 640x640, Early stopping)
5. **Quantitative & Qualitative Evaluation** (Precision, Recall, mAP50, mAP50-95, Latency)
6. **Error Analysis & Visual Diagnostics** (False positives/negatives inspection)
7. **ONNX Export & Parity Verification** (`pothole.onnx` for browser deployment)
8. **Download Artifacts** (`models/best.pt`, `models/pothole.onnx`)

## 1. Environment Setup & GPU Check

In [ ]:
# Verify GPU runtime
!nvidia-smi

# Install required packages
!pip install -q ultralytics onnx onnxruntime onnxslim opencv-python matplotlib pyyaml

In [ ]:
import torch
import ultralytics
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics Version: {ultralytics.__version__}")

## 2. Dataset Acquisition

Choose **Option A** (Direct Download via Roboflow / Public Dataset) or **Option B** (Google Drive mount for your custom dataset / RDD2022).

In [ ]:
# OPTION A: Download standard pre-annotated pothole dataset (Roboflow YOLO format)
# Replace with your Roboflow API key or direct dataset download if available:
'''
!pip install -q roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("pothole-detection").project("pothole-detection-system")
version = project.version(1)
dataset = version.download("yolov8")
'''

# OPTION B: Mount Google Drive if dataset.zip is in your Drive
from google.colab import drive
import os
import shutil

if not os.path.exists('/content/dataset'):
    drive.mount('/content/drive')
    # Assuming dataset.zip is in Google Drive root or specified folder:
    if os.path.exists('/content/drive/MyDrive/dataset.zip'):
        !unzip -q /content/drive/MyDrive/dataset.zip -d /content/
        print("Dataset unzipped successfully to /content/dataset")

## 3. Dataset Validation & Integrity Check

Validate coordinate normalization, single class ID (`0: pothole`), check for corrupt images and verify no leaks.

In [ ]:
import os
import yaml
from pathlib import Path
from PIL import Image

# Ensure data.yaml is properly configured for single-class pothole
data_yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: pothole
"""

os.makedirs('/content/dataset', exist_ok=True)
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print("data.yaml verified:")
!cat /content/dataset/data.yaml

In [ ]:
# Dataset Statistics & Validation Function
def inspect_dataset(data_dir="/content/dataset"):
    base = Path(data_dir)
    for split in ["train", "val", "test"]:
        img_dir = base / "images" / split
        lbl_dir = base / "labels" / split
        if not img_dir.exists():
            print(f"[-] Split {split} not found.")
            continue
        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
        lbls = list(lbl_dir.glob("*.txt"))
        
        box_count = 0
        for lf in lbls:
            with open(lf, 'r') as f:
                lines = [l.strip() for l in f if l.strip()]
                box_count += len(lines)
        
        print(f"[{split.upper()}] Images: {len(imgs)} | Labels: {len(lbls)} | Bounding Boxes: {box_count}")

inspect_dataset()

## 4. Train YOLO11n Baseline Detector

Training specifications:
- Architecture: `yolo11n.pt` (Nano model optimized for mobile)
- Resolution: `640x640`
- Batch Size: `16`
- Epochs: `100` (with early stopping `patience=20`)
- Augmentations: Road lighting variations (`hsv_s`, `hsv_v`), horizontal flipping, slight tilt

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11n weights
try:
    model = YOLO('yolo11n.pt')
except Exception as e:
    print(f"Falling back to yolov8n: {e}")
    model = YOLO('yolov8n.pt')

# Train model
train_results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    deterministic=True,
    project='/content/runs/pothole_training',
    name='yolo11n_pothole_v1',
    # Augmentation tuning for bike-mounted road perspectives
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    save=True,
    plots=True
)

## 5. Comprehensive Quantitative Evaluation

In [ ]:
# Evaluate on Validation set
val_metrics = model.val(data='/content/dataset/data.yaml', split='val', imgsz=640, conf=0.35, iou=0.5)

print("=== VALIDATION METRICS ===")
print(f"Precision:    {val_metrics.box.mp:.4f}")
print(f"Recall:       {val_metrics.box.mr:.4f}")
print(f"mAP@0.50:     {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {val_metrics.box.map:.4f}")
print(f"Inference:    {val_metrics.speed['inference']:.2f} ms per frame")

In [ ]:
# Evaluate on Unseen Test set
if os.path.exists('/content/dataset/images/test'):
    test_metrics = model.val(data='/content/dataset/data.yaml', split='test', imgsz=640, conf=0.35, iou=0.5)
    print("=== TEST SET METRICS ===")
    print(f"Test Precision:    {test_metrics.box.mp:.4f}")
    print(f"Test Recall:       {test_metrics.box.mr:.4f}")
    print(f"Test mAP@0.50:     {test_metrics.box.map50:.4f}")
    print(f"Test mAP@0.5:0.95: {test_metrics.box.map:.4f}")

## 6. Visual Evaluation & Error Analysis

Display sample prediction images with bounding boxes and confidence scores.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Run prediction on test images
test_imgs = glob.glob('/content/dataset/images/test/*.jpg')[:6]
if not test_imgs:
    test_imgs = glob.glob('/content/dataset/images/val/*.jpg')[:6]

pred_results = model.predict(source=test_imgs, imgsz=640, conf=0.35, save=True, project='/content/runs/predictions')

# Render results inline
saved_preds = glob.glob('/content/runs/predictions/**/*.jpg', recursive=True)[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, ax in enumerate(axes.flat):
    if idx < len(saved_preds):
        img = Image.open(saved_preds[idx])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Sample {idx+1}")
plt.tight_layout()
plt.show()

## 7. ONNX Export & Sanity Verification (`pothole.onnx`)

Exports the model for browser inference using ONNX Runtime Web, verifies input/output tensors, and tests numerical parity.

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

# Export best PyTorch weights to ONNX
best_weights = '/content/runs/pothole_training/yolo11n_pothole_v1/weights/best.pt'
best_model = YOLO(best_weights)
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True, opset=17)

print(f"Exported ONNX model path: {onnx_path}")

# Inspect ONNX structure with ONNX Runtime
session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_meta = session.get_inputs()[0]
output_meta = session.get_outputs()[0]

print("\n=== ONNX MODEL SPECIFICATIONS ===")
print(f"• Input Name:   {input_meta.name}")
print(f"• Input Shape:  {input_meta.shape}")
print(f"• Input Type:   {input_meta.type}")
print(f"• Output Name:  {output_meta.name}")
print(f"• Output Shape: {output_meta.shape}")
print(f"• Output Type:  {output_meta.type}")
print(f"• File Size:    {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")

# Sanity check with random tensor
dummy = np.random.uniform(0.0, 1.0, (1, 3, 640, 640)).astype(np.float32)
out = session.run(None, {input_meta.name: dummy})[0]
print(f"\n[+] Sanity inference successful! Output shape: {out.shape}")

## 8. Download Trained Artifacts

Download `best.pt` and `pothole.onnx` to place directly in your local `D:\Pothhole\models\` folder.

In [ ]:
from google.colab import files
import shutil

# Copy to clean names
shutil.copy2(best_weights, '/content/best.pt')
shutil.copy2(onnx_path, '/content/pothole.onnx')

# Download to local machine
files.download('/content/best.pt')
files.download('/content/pothole.onnx')
print("[+] Download triggered for best.pt and pothole.onnx!")